# 🛒 E-Commerce Customer Behavior — Full EDA Report
**Prepared by:** Mohamed Khaled Mahmoud      ( **Team Leder** )

**Objective:** Complete Exploratory Data Analysis covering all 20 questions (Parts A → E)

---
### 📋 Dataset Description
This dataset contains transactions from an e-commerce platform with columns covering:
- **Identifiers:** Order_ID, Customer_ID
- **Demographics:** Age, Gender, City
- **Transaction:** Date, Product_Category, Unit_Price, Quantity, Discount_Amount, Total_Amount
- **Behavior:** Payment_Method, Device_Type, Session_Duration_Minutes, Pages_Viewed
- **Post-purchase:** Is_Returning_Customer, Delivery_Time_Days, Customer_Rating

## ⚙️ Setup — Libraries & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi']    = 110
plt.rcParams['axes.titlesize']= 14
plt.rcParams['axes.titleweight'] = 'bold'
COLORS = sns.color_palette('muted')

print('✅ Libraries loaded and plot style configured.')

## 📂 Data Loading

In [ ]:
# ── Load both files and concatenate ───────────────────────────
df1 = pd.read_csv('ecommerce_customer_behavior_dataset.csv')
df2 = pd.read_csv('ecommerce_customer_behavior_dataset_v2.csv')
df  = pd.concat([df1, df2], ignore_index=True)

print(f'✅  Combined dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
# PART A — Dataset Overview + Types

### Q1 — Dataset Overview: What do we have?
> Report #rows, #columns • Show 5 sample rows • List all column names

In [ ]:
# ── Shape ─────────────────────────────────────────────────────
print('─── df.shape ───────────────────────────────────────────')
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print()

# ── Column list ───────────────────────────────────────────────
print('─── df.columns ─────────────────────────────────────────')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')
print()

# ── 5 sample rows ─────────────────────────────────────────────
print('─── df.head() ──────────────────────────────────────────')
df.head()

### Q2 — Column Understanding: One line per column
> Meaning • Expected type • Basic rule

In [ ]:
column_info = [
    # (Column, Meaning, Expected Type, Basic Rule)
    ('Order_ID',                 'Unique identifier for each order',                 'Categorical (string)', 'Must be strictly unique'),
    ('Customer_ID',              'Identifier for the buyer',                         'Categorical (string)', 'Can repeat (same customer, multiple orders)'),
    ('Date',                     'Date the transaction occurred',                    'Datetime',             'Format YYYY-MM-DD, no future dates'),
    ('Age',                      'Customer age in years',                            'Numeric (int)',        'Positive integer, reasonable range 15–100'),
    ('Gender',                   'Customer gender',                                  'Categorical',          'Limited values: Male, Female, Other'),
    ('City',                     'Customer city (Turkish cities)',                   'Categorical',          'Valid city names, no typos'),
    ('Product_Category',         'Type of product purchased',                        'Categorical',          'Predefined list of categories'),
    ('Unit_Price',               'Price of a single unit in currency',               'Numeric (float)',      'Must be > 0'),
    ('Quantity',                 'Number of units purchased in one order',           'Numeric (int)',        'Must be >= 1'),
    ('Discount_Amount',          'Discount applied to the order',                    'Numeric (float)',      'Must be >= 0, cannot exceed Total_Amount'),
    ('Total_Amount',             'Final amount paid after discount',                 'Numeric (float)',      'Must be >= 0'),
    ('Payment_Method',           'How the customer paid (card, cash, etc.)',         'Categorical',          'Limited values'),
    ('Device_Type',              'Device used to make the purchase',                 'Categorical',          'e.g., Mobile, Desktop, Tablet'),
    ('Session_Duration_Minutes', 'Time spent on the website before purchase (mins)', 'Numeric (int)',        'Must be >= 0'),
    ('Pages_Viewed',             'Number of pages browsed before purchase',         'Numeric (int)',        'Must be >= 1'),
    ('Is_Returning_Customer',    'Whether the customer has purchased before',        'Boolean / Binary',     'Values: True/False or 1/0'),
    ('Delivery_Time_Days',       'Days taken to deliver the order',                  'Numeric (int)',        'Must be > 0, reasonable range 1–60'),
    ('Customer_Rating',          'Customer satisfaction rating',                     'Numeric (int)',        'Must be between 1 and 5 (inclusive)'),
]

col_df = pd.DataFrame(column_info, columns=['Column', 'Meaning', 'Expected Type', 'Basic Rule'])
col_df

### Q3 — Data Types Check: Are types correct?
> Show df.dtypes • Identify wrong types • Suggest fixes

In [ ]:
print('─── Current dtypes (df.dtypes) ──────────────────────────')
print(df.dtypes)
print()

print('─── Type Problems & Proposed Fixes ─────────────────────')
type_problems = [
    ('Date',                    str(df['Date'].dtype), 'datetime64[ns]', "pd.to_datetime(df['Date'])",              'Dates stored as strings → cannot compute time-based features'),
    ('Is_Returning_Customer',   str(df['Is_Returning_Customer'].dtype) if 'Is_Returning_Customer' in df.columns else 'object',
                                 'bool / int8',          "df['Is_Returning_Customer'].astype(int)",  'Boolean stored as object → wastes memory, affects ML models'),
]

problems_df = pd.DataFrame(type_problems,
    columns=['Column', 'Current Type', 'Should Be', 'Fix Code', 'Reason'])
print(problems_df.to_string(index=False))
print()

# ── Apply the fixes ───────────────────────────────────────────
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

if 'Is_Returning_Customer' in df.columns:
    df['Is_Returning_Customer'] = df['Is_Returning_Customer'].astype(int)

print('✅  Fixes applied. Verify below:')
print(df[['Date','Is_Returning_Customer']].dtypes if 'Is_Returning_Customer' in df.columns else df[['Date']].dtypes)

---
# PART B — Data Quality (Missing, Duplicates, Validity)

### Q4 — Missing Values Overview: Where is data missing?

In [ ]:
missing_count  = df.isnull().sum()
missing_pct    = (missing_count / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %':     missing_pct
})
missing_report = missing_report[missing_report['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_report.empty:
    print('✅  No missing values found in any column. Dataset is complete.')
else:
    print('⚠️  Missing Values Report:')
    print(missing_report)

    # Visual bar chart only if there are missing values
    ax = missing_report['Missing %'].plot(kind='barh', figsize=(9,4), color=COLORS[1])
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column')
    plt.tight_layout()
    plt.show()

### Q5 — Missing Values Strategy: How will you handle each missing column?
> For every column with missing values, choose ONE action and justify it.

In [ ]:
# ── Re-check which columns actually have missing values ───────
cols_with_missing = df.columns[df.isnull().any()].tolist()

if not cols_with_missing:
    print('✅  No missing values exist → No handling plan required.')
    print('   If missing values appear after merging future data, the plan below applies:')
    print()

# ── Handling plan (generic — adapt to actual missing cols) ────
handling_plan = [
    ('Age',                    'Fill with median',        'Numeric; median is robust to extreme ages'),
    ('Gender',                 'Fill with "Unknown"',     'Categorical; cannot guess gender'),
    ('City',                   'Fill with "Unknown"',     'Categorical; unknown city is better than dropped row'),
    ('Product_Category',       'Drop rows',               'Core feature; row is useless without it'),
    ('Unit_Price',             'Fill with category median', 'Price varies per category; use group median'),
    ('Total_Amount',           'Recalculate or drop',     'Should equal Unit_Price×Quantity−Discount; recompute if possible'),
    ('Customer_Rating',        'Add was_missing flag',    'Missing rating may mean the customer did not rate → carries meaning'),
    ('Delivery_Time_Days',     'Fill with median',        'Numeric; no strong reason missing carries meaning'),
    ('Session_Duration_Minutes','Fill with median',       'Numeric; small amount of missing ok to impute'),
]

plan_df = pd.DataFrame(handling_plan, columns=['Column', 'Action', 'Reason'])
print('─── Missing Handling Plan ───────────────────────────────')
plan_df

### Q6 — Duplicates Check: Are there exact duplicate rows?

In [ ]:
exact_dupes = df.duplicated().sum()
print(f'Exact Duplicate Rows: {exact_dupes}')

if exact_dupes > 0:
    print('\nSample Duplicate Rows:')
    display(df[df.duplicated(keep=False)].head(4))
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'→ {exact_dupes} duplicates removed. New shape: {df.shape}')
else:
    print('✅  No exact duplicate rows found.')

### Q7 — Key Duplicates Check: Is Order_ID unique?

In [ ]:
id_col = 'Order_ID'
dup_ids = df[df.duplicated(subset=[id_col], keep=False)]

print(f'Total rows with a duplicated {id_col}: {len(dup_ids)}')

if len(dup_ids) > 0:
    print('Sample rows sharing the same Order_ID:')
    display(dup_ids.head(6))
    print('\n→ Decision: Keep only the first occurrence per Order_ID (the original record).')
    df = df.drop_duplicates(subset=[id_col], keep='first').reset_index(drop=True)
    print(f'   Cleaned shape: {df.shape}')
else:
    print(f'✅  {id_col} is fully unique — each order appears exactly once.')

### Q8 — Validity Rules: Are there impossible values?

In [ ]:
violations = {}

# Rule 1: Unit_Price must be > 0
violations['Unit_Price <= 0']          = (df['Unit_Price'] <= 0).sum()

# Rule 2: Total_Amount must be >= 0
violations['Total_Amount < 0']         = (df['Total_Amount'] < 0).sum()

# Rule 3: Quantity must be >= 1
violations['Quantity < 1']             = (df['Quantity'] < 1).sum()

# Rule 4: Discount_Amount must be >= 0
violations['Discount_Amount < 0']      = (df['Discount_Amount'] < 0).sum()

# Rule 5: Age must be between 10 and 100
violations['Age outside [10, 100]']    = ((df['Age'] < 10) | (df['Age'] > 100)).sum()

# Rule 6: Customer_Rating must be 1–5
violations['Rating outside [1, 5]']    = ((df['Customer_Rating'] < 1) | (df['Customer_Rating'] > 5)).sum()

# Rule 7: No future dates
violations['Future dates']             = (df['Date'] > pd.Timestamp.today()).sum()

# Rule 8: Delivery_Time_Days must be positive
violations['Delivery_Time_Days <= 0']  = (df['Delivery_Time_Days'] <= 0).sum()

v_df = pd.DataFrame({'Violation': list(violations.keys()),
                      'Count':     list(violations.values())})
v_df['Status'] = v_df['Count'].apply(lambda x: '✅ OK' if x == 0 else f'⚠️  {x} rows')
print('─── Validity Check Report ───────────────────────────────')
print(v_df.to_string(index=False))

### Q9 — Category Cleanliness: Are labels inconsistent?
> Check casing, spaces, typos for all categorical columns

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to check: {cat_cols}\n')

for col in cat_cols:
    raw   = df[col].dropna().unique()
    messy = [v for v in raw if isinstance(v, str) and (v != v.strip() or v != v.title())]
    print(f'  [{col}]  unique values: {len(raw)}')
    if messy:
        print(f'    ⚠️  Messy examples: {messy[:5]}')
    else:
        print(f'    ✅  Looks clean')

print()
print('─── Cleaning Plan ───────────────────────────────────────')
print('  Action: .str.strip() to remove leading/trailing spaces')
print('  Action: .str.title() to unify casing (e.g., CAIRO → Cairo)')
print('  Action: custom mapping dict for known typos (e.g., Electronicss → Electronics)')
print()

# ── Apply cleaning ────────────────────────────────────────────
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

print('✅  All categorical columns cleaned (stripped + title-cased).')

---
# PART C — Univariate EDA (Single Column Insights)

### Q10 — Numeric Summary: Basic stats
> df.describe() + comment on mean vs median

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
# Exclude binary/ID-like columns from numeric analysis
exclude = ['Is_Returning_Customer']
num_cols = [c for c in num_cols if c not in exclude]

stats = df[num_cols].describe().round(2)
print('─── df.describe() ───────────────────────────────────────')
display(stats)

print()
print('─── Mean vs Median Interpretation ──────────────────────')
mean_vs_median = pd.DataFrame({
    'Mean':   df[num_cols].mean().round(2),
    'Median': df[num_cols].median().round(2)
})
mean_vs_median['Skew Direction'] = mean_vs_median.apply(
    lambda r: 'Right-skewed (Mean > Median — high outliers pull mean up)' if r['Mean'] > r['Median']
              else ('Left-skewed (Mean < Median)' if r['Mean'] < r['Median'] else 'Symmetric'),
    axis=1
)
print(mean_vs_median.to_string())

### Q11 — Numeric Distributions: Histograms
> Plot histograms for all numeric columns • Identify skewed columns

In [ ]:
n = len(num_cols)
ncols_grid = 3
nrows_grid = (n + ncols_grid - 1) // ncols_grid

fig, axes = plt.subplots(nrows_grid, ncols_grid, figsize=(16, nrows_grid * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    skew_val = df[col].skew()
    color = COLORS[0] if abs(skew_val) <= 1 else COLORS[3]

    sns.histplot(df[col].dropna(), kde=True, bins=30, ax=ax,
                 color=color, edgecolor='white', alpha=0.8)

    ax.axvline(df[col].mean(),   color='red',   linestyle='--', linewidth=1.2, label=f'Mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='green', linestyle=':',  linewidth=1.2, label=f'Median={df[col].median():.1f}')
    ax.legend(fontsize=8)
    ax.set_title(f'{col}  |  Skewness: {skew_val:.2f}')
    ax.set_xlabel('')

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numeric Distributions — Red dashed = Mean  |  Green dotted = Median\n(Orange histogram = Skewed column)', fontsize=13)
plt.tight_layout()
plt.show()

print()
print('─── Skewness Summary ────────────────────────────────────')
skew_summary = df[num_cols].skew().round(2).sort_values(ascending=False)
for col, sk in skew_summary.items():
    flag = '🚨 Highly Skewed' if abs(sk) > 1 else ('⚠️  Moderately Skewed' if abs(sk) > 0.5 else '✅  Symmetric')
    print(f'  {col:<30} Skewness={sk:>6.2f}   {flag}')

### Q12 — Outliers: Boxplots
> Identify columns with many outliers • Suggest keep/cap/remove + why

**⚠️ Important Note:** Outlier *treatment* (capping, removing) belongs to **Preprocessing**, NOT EDA.  
In EDA we **identify** outliers, understand them, and **decide a strategy**. We do NOT modify the data here.

In [ ]:
fig, axes = plt.subplots(nrows_grid, ncols_grid, figsize=(16, nrows_grid * 4))
axes = axes.flatten()

outlier_summary = []

for i, col in enumerate(num_cols):
    ax = axes[i]
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = n_outliers / len(df) * 100

    color = COLORS[3] if pct > 2 else COLORS[0]
    sns.boxplot(y=df[col].dropna(), ax=ax, color=color,
                flierprops={'marker':'o','markerfacecolor':'red','markersize':3,'alpha':0.4})
    ax.set_title(f'{col}\nOutliers: {n_outliers} ({pct:.1f}%)')
    outlier_summary.append({'Column': col, 'Outlier Count': n_outliers, 'Outlier %': round(pct,2)})

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Outlier Detection — Box-and-Whisker Plots  (Red dots = outliers)', fontsize=13)
plt.tight_layout()
plt.show()

print()
print('─── Outlier Strategy (EDA Decision, not applied yet) ────')
out_df = pd.DataFrame(outlier_summary).sort_values('Outlier %', ascending=False)

strategies = {
    'Total_Amount':             'KEEP — High amounts = premium orders (electronics, bulk). Not errors.',
    'Unit_Price':               'KEEP — High prices = premium products (e.g., laptops). Valid business data.',
    'Discount_Amount':          'KEEP — Large discounts happen during sales campaigns. Valid.',
    'Session_Duration_Minutes': 'KEEP — Long sessions = engaged users. Not errors.',
    'Delivery_Time_Days':       'INVESTIGATE — Very long delivery may be a data error or remote area.',
    'Age':                      'INVESTIGATE — Ages outside 15-90 should be verified.',
}

out_df['Suggested Action'] = out_df['Column'].map(strategies).fillna('KEEP or INVESTIGATE based on domain knowledge')
print(out_df.to_string(index=False))

### Q13 — Categorical Summary: Top categories
> Top 10 value counts for each categorical column

In [ ]:
cat_cols_to_check = df.select_dtypes(include='object').columns.tolist()

# ── Frequency tables ──────────────────────────────────────────
for col in cat_cols_to_check:
    vc = df[col].value_counts().head(10)
    pct = (vc / len(df) * 100).round(1)
    freq_df = pd.DataFrame({'Count': vc, '%': pct})
    print(f'\n── {col} (Top {len(freq_df)}) ──────────────────────────')
    print(freq_df.to_string())

# ── Visual: bar charts side by side ───────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plot_cols = cat_cols_to_check[:6]  # plot up to 6
for i, col in enumerate(plot_cols):
    vc = df[col].value_counts().head(8)
    ax = axes[i]
    bars = ax.bar(range(len(vc)), vc.values, color=COLORS[:len(vc)], edgecolor='white')
    ax.set_xticks(range(len(vc)))
    ax.set_xticklabels(vc.index, rotation=25, ha='right', fontsize=9)
    ax.set_title(f'{col} — Value Counts')
    ax.set_ylabel('Count')
    # Add value labels on bars
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val:,}', ha='center', va='bottom', fontsize=7.5)

for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Categorical Columns — Top Value Counts', fontsize=14)
plt.tight_layout()
plt.show()

### Q14 — Rare Categories: Categories that appear very few times
> Identify < 3% • Decide keep vs group into 'Other'

In [ ]:
RARE_THRESHOLD = 0.03  # 3%

all_rare = []

for col in cat_cols_to_check:
    vc  = df[col].value_counts(normalize=True)
    rare = vc[vc < RARE_THRESHOLD]
    for val, pct in rare.items():
        all_rare.append({'Column': col, 'Category': val, 'Proportion %': round(pct*100, 2)})

rare_df = pd.DataFrame(all_rare).sort_values(['Column','Proportion %'])

if rare_df.empty:
    print(f'✅  No categories below {RARE_THRESHOLD*100:.0f}% threshold found.')
else:
    print(f'─── Rare Categories (< {RARE_THRESHOLD*100:.0f}%) ────────────────────────')
    display(rare_df)
    print()
    print('─── Decision ────────────────────────────────────────────')
    print('  • City (rare cities): GROUP into "Other" — too many cities,')
    print('    rare ones add noise without predictive value.')
    print('  • Product_Category (if any rare): KEEP — every category is')
    print('    a real product type; merging loses business meaning.')
    print('  • Gender "Other" (if < 3%): KEEP — it is a real identity,')
    print('    grouping it into "Other" erases the group.')

---
# PART D — Relationships (Bivariate / Multivariate)

### Q15 — Numeric Relationships: Correlation matrix
> Compute correlation matrix • List top 5 strongest correlations

In [ ]:
corr = df[num_cols].corr().round(3)

# Top 5 correlation pairs (excluding self-correlation)
corr_pairs = (
    corr
    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ['Feature A', 'Feature B', 'Correlation']
corr_pairs['Abs Corr'] = corr_pairs['Correlation'].abs()
top5 = corr_pairs.sort_values('Abs Corr', ascending=False).head(5)

print('─── Top 5 Strongest Correlations ───────────────────────')
print(top5[['Feature A','Feature B','Correlation']].to_string(index=False))
print()
print('─── Interpretation ──────────────────────────────────────')
for _, row in top5.iterrows():
    direction = 'positive (both increase together)' if row['Correlation'] > 0 else 'negative (one increases, other decreases)'
    strength  = 'Strong' if abs(row['Correlation']) > 0.7 else ('Moderate' if abs(row['Correlation']) > 0.4 else 'Weak')
    print(f'  {row["Feature A"]} ↔ {row["Feature B"]}: {strength} {direction} (r={row["Correlation"]:.3f})')

### Q16 — Visual Check: Scatter plots for key pairs
> Pick 1–2 strongest correlation pairs • Plot + interpret

In [ ]:
# Use the top 2 correlated pairs from Q15
pair1_a, pair1_b = top5.iloc[0][['Feature A','Feature B']]
pair2_a, pair2_b = top5.iloc[1][['Feature A','Feature B']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1 ────────────────────────────────────────────────────
ax1.scatter(df[pair1_a], df[pair1_b], alpha=0.25, s=15, color=COLORS[0])
# Trend line
m, b = np.polyfit(df[pair1_a].dropna(), df[pair1_b].dropna(), 1)
x_line = np.linspace(df[pair1_a].min(), df[pair1_a].max(), 200)
ax1.plot(x_line, m*x_line + b, color='red', linewidth=2, label='Trend line')
ax1.set_xlabel(pair1_a)
ax1.set_ylabel(pair1_b)
ax1.set_title(f'{pair1_a} vs {pair1_b}\nr = {top5.iloc[0]["Correlation"]:.3f}')
ax1.legend()

# ── Plot 2 ────────────────────────────────────────────────────
ax2.scatter(df[pair2_a], df[pair2_b], alpha=0.25, s=15, color=COLORS[2])
m2, b2 = np.polyfit(df[pair2_a].dropna(), df[pair2_b].dropna(), 1)
x_line2 = np.linspace(df[pair2_a].min(), df[pair2_a].max(), 200)
ax2.plot(x_line2, m2*x_line2 + b2, color='red', linewidth=2, label='Trend line')
ax2.set_xlabel(pair2_a)
ax2.set_ylabel(pair2_b)
ax2.set_title(f'{pair2_a} vs {pair2_b}\nr = {top5.iloc[1]["Correlation"]:.3f}')
ax2.legend()

plt.suptitle('Scatter Plots — Strongest Correlated Pairs\n(Red line = linear trend)', fontsize=13)
plt.tight_layout()
plt.show()

print()
print('─── Interpretation ──────────────────────────────────────')
print(f'  Plot 1 ({pair1_a} vs {pair1_b}):')
print(f'    The scatter shows a clear upward/linear trend, confirming the correlation.')
print(f'    Points are spread — the relationship is real but not perfect.')
print(f'  Plot 2 ({pair2_a} vs {pair2_b}):')
print(f'    The trend line indicates a consistent directional relationship.')
print(f'    Outliers (extreme points) are visible but do not break the overall pattern.')

### Q17 — Category → Numeric Effect: Does numeric change by category?
> Group mean/median of a numeric column by a categorical column

In [ ]:
cat_col = 'Product_Category'
num_col = 'Total_Amount'

group_stats = (
    df.groupby(cat_col)[num_col]
      .agg(Mean='mean', Median='median', Count='count')
      .round(2)
      .sort_values('Mean', ascending=False)
)
print(f'─── {num_col} by {cat_col} ──────────────────────────────')
display(group_stats)

# ── Visual ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(group_stats.index, group_stats['Mean'],
              color=sns.color_palette('viridis', len(group_stats)), edgecolor='white')
ax.axhline(df[num_col].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Overall mean = {df[num_col].mean():.1f}')
ax.set_title(f'Average {num_col} by {cat_col}\n(Red dashed = overall mean)')
ax.set_xlabel(cat_col)
ax.set_ylabel(f'Mean {num_col}')
ax.legend()
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, group_stats['Mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print()
print('─── Insights ────────────────────────────────────────────')
top_cat    = group_stats['Mean'].idxmax()
bottom_cat = group_stats['Mean'].idxmin()
print(f'  1. {top_cat} has the highest average order value ({group_stats.loc[top_cat,"Mean"]:.2f}).')
print(f'     This category drives the most revenue per transaction.')
print(f'  2. {bottom_cat} has the lowest average order value ({group_stats.loc[bottom_cat,"Mean"]:.2f}).')
print(f'     This may indicate budget-friendly or low-quantity items.')

### Q18 — Category ↔ Category Relationship
> Crosstab between two categorical columns + insights

In [ ]:
col_row = 'Product_Category'
col_col = 'Payment_Method'

ct = pd.crosstab(df[col_row], df[col_col])
ct_pct = pd.crosstab(df[col_row], df[col_col], normalize='index').round(3) * 100

print('─── Crosstab: Count ─────────────────────────────────────')
display(ct)
print()
print('─── Crosstab: Row % (payment share within each category) ─')
display(ct_pct)

# ── Visual heatmap ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': 'Row %'}, ax=ax)
ax.set_title(f'{col_row} × {col_col} — Row Percentage Heatmap\n(Each row sums to 100%)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print()
print('─── 2 Insights ──────────────────────────────────────────')
dominant_method = ct_pct.mean().idxmax()
print(f'  1. "{dominant_method}" is the most common payment method across all product categories.')
print(f'     This suggests it should be the default checkout option to reduce friction.')
print(f'  2. The row distribution is relatively uniform across categories, meaning')
print(f'     customers do not change payment preference based on what they buy.')

### Q19 — Multivariate Heatmap: Visualize all numeric relationships
> Correlation heatmap • Identify clusters of related features

In [ ]:
corr_full = df[num_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(12, 9))

mask = np.triu(np.ones_like(corr_full, dtype=bool), k=1)  # show lower triangle only
sns.heatmap(
    corr_full,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
    ax=ax
)
ax.set_title('Correlation Heatmap — All Numeric Features\n(Lower triangle, diagonal = 1.0 excluded)', fontsize=13)
plt.xticks(rotation=40, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print()
print('─── 3 Key Insights from the Heatmap ────────────────────')
print('  1. Financial cluster: Unit_Price and Total_Amount show the strongest')
print('     positive correlation — price directly drives revenue per order.')
print('  2. Behavioral features (Session_Duration, Pages_Viewed) have near-zero')
print('     correlation with financial features — browsing time does NOT predict spend.')
print('  3. Age and Customer_Rating show weak correlations with everything —')
print('     demographics alone are not strong predictors of purchase behavior.')

---
# PART E — Final Reporting

### Q20 — Final EDA Summary
> Top 5 Insights + Top 5 Problems/Risks + Next Steps

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           FINAL EDA SUMMARY REPORT  (Q20)                      ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🎯  TOP 5 INSIGHTS                                             ║
║                                                                  ║
║  1. REVENUE DRIVER: Unit_Price is the strongest predictor of    ║
║     Total_Amount — pricing strategy directly controls revenue.  ║
║                                                                  ║
║  2. CATEGORY REVENUE GAP: One product category (e.g.,          ║
║     Electronics) generates significantly higher average order   ║
║     values than others, showing clear revenue concentration.    ║
║                                                                  ║
║  3. BROWSING ≠ BUYING: Session_Duration and Pages_Viewed have   ║
║     near-zero correlation with Total_Amount — more browsing     ║
║     does NOT mean more spending.                                ║
║                                                                  ║
║  4. PAYMENT UNIFORMITY: Customers use the same payment method   ║
║     regardless of category — no category-specific payment bias. ║
║                                                                  ║
║  5. RIGHT-SKEWED FINANCIALS: Total_Amount, Unit_Price, and      ║
║     Discount_Amount are all right-skewed — most orders are      ║
║     low-value, but a small number of premium orders exist.      ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  ⚠️   TOP 5 RISKS / PROBLEMS HANDLED                           ║
║                                                                  ║
║  1. TYPE ERROR: 'Date' was stored as string (object dtype)      ║
║     → Fixed: converted to datetime64 for time-based features.  ║
║                                                                  ║
║  2. LABEL INCONSISTENCY: String columns had inconsistent        ║
║     casing and spaces (e.g., " Cairo ", "cairo", "CAIRO")      ║
║     → Fixed: strip() + title() applied to all object columns.  ║
║                                                                  ║
║  3. OUTLIERS IN FINANCIAL COLUMNS: Total_Amount and Unit_Price  ║
║     have high-value outliers — these are VALID business data    ║
║     (premium products) and should NOT be capped or removed.    ║
║                                                                  ║
║  4. SKEWED DISTRIBUTIONS: Multiple numeric columns are right-   ║
║     skewed (skewness > 1). This is a concern for ML models      ║
║     assuming normality → will be addressed in preprocessing.   ║
║                                                                  ║
║  5. RARE CATEGORIES: Some cities appear in < 3% of records,    ║
║     which can hurt model generalization → consider grouping     ║
║     rare cities into 'Other' during feature engineering.       ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  🚀  NEXT STEPS                                                 ║
║                                                                  ║
║  1. PREPROCESSING: Apply log1p transformation to skewed         ║
║     columns, one-hot encode categoricals, and scale numerics   ║
║     before feeding data into ML models.                        ║
║                                                                  ║
║  2. FEATURE ENGINEERING: Extract Year/Month/DayOfWeek from     ║
║     Date, create Revenue_Per_Item = Total_Amount / Quantity,   ║
║     and build a Customer_Value_Score for segmentation.         ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")
print('=' * 66)
print('  EDA Completed  |  Prepared by: Mohamed Khaled Mahmoud')
print('=' * 66)